# WFP Distance Replication From `README_distance_replication.txt`

This notebook implements the distance-computation replication described in:

`C:\local\GIT\Public-Infrastructure-Service-Access-remove-wfpexplaining\README_distance_replication.txt`

The README describes the last-mile step used for the WFP slide deck: start from already-geocoded school routing targets, generate candidate facility points on a 2 km grid, snap both layers to the Nusa Tenggara road network, and write a split candidate-to-school road-distance matrix plus manifest files.

The notebook uses the reengineered `scalable_distances` package for shared-cache source resolution, OSM network loading, router-neutral network tables, and split matrix persistence. It also carries the updated README guidance for the next layers: facility-location assignment and per-facility TSP diagnostics.

The full run is intentionally guarded by switches. The full README configuration can be very large: roughly 14,101 school destinations and a dense 2 km candidate grid over the Nusa Tenggara bbox. By default this notebook executes prechecks and candidate generation only; set `RUN_FULL_DISTANCE_MATRIX = True` when you want to launch the country-scale matrix computation.

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from hashlib import sha256
from pathlib import Path
import json
import math
import sys
from typing import Iterable

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src" / "scalable_distances").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_ROOT = PROJECT_ROOT / "src"
assert (SRC_ROOT / "scalable_distances").exists(), f"Could not find scalable_distances under {PROJECT_ROOT}"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import yaml
from pyproj import Transformer
from scipy.spatial import cKDTree

from scalable_distances.config import CountryDataSources
from scalable_distances.io import download_if_missing
from scalable_distances.matrix import write_matrix_outputs
from scalable_distances.network import load_driving_network
from scalable_distances.routing.base import NetworkData

## Exact README Configuration

This cell records the README command in a structured form. The same values are used by the loader, candidate generator, snapper, matrix writer, and manifest.

README command being replicated:

```powershell
py run_pipeline.py nusa_tenggara --sources candidates --destinations table `
  --destination-table "C:\local\GIT\route-the-meals\geocoding\draft\17_routing_targets_enhanced.csv" `
  --destination-lon-column routing_lon --destination-lat-column routing_lat --destination-id-column source_id `
  --bbox 118.8 -11.1 125.4 -7.1 `
  --candidate-grid-spacing-m 2000 --candidate-max-snap-dist-m 1000 --max-total-dist 150000 `
  --amenity school --aggregate-factor 10 --save-map --map-basemap voyager-no-labels --matrix-output-mode split
```

In [ ]:
@dataclass(frozen=True)
class DistanceReplicationConfig:
    country_slug: str = "nusa-tenggara"
    iso3: str = "IDN"
    geofabrik_region: str = "asia/indonesia"
    pbf_filename: str = "nusa-tenggara-latest.osm.pbf"
    base_dir: Path = Path(r"C:\local\Download_Depot\indonesia_nusa_tenggara_data")
    destination_table: Path = Path(r"C:\local\GIT\route-the-meals\geocoding\draft\17_routing_targets_enhanced.csv")
    destination_lon_column: str = "routing_lon"
    destination_lat_column: str = "routing_lat"
    destination_id_column: str = "source_id"
    bbox: tuple[float, float, float, float] = (118.8, -11.1, 125.4, -7.1)
    candidate_grid_spacing_m: int = 2000
    candidate_max_snap_dist_m: int = 1000
    max_total_dist_m: int = 150000
    amenity: str = "school"
    aggregate_factor: int = 10
    matrix_output_mode: str = "split"
    map_basemap: str = "voyager-no-labels"
    metric_crs: str = "EPSG:32751"  # UTM 51S; sufficient for NTT replication diagnostics.
    run_tag: str = "pop_1_sample_1.0_max_all_agg_10_maxdist_150000_amenity_school_candidates_spacing_2000_maxsnap_1000"

CFG = DistanceReplicationConfig()
OUTPUT_DIR = CFG.base_dir / "outputs"
CACHE_DIR = CFG.base_dir / "cache"
SMOKE_OUTPUT_DIR = OUTPUT_DIR / "notebook_smoke"

# Safety switches. The full matrix is intentionally disabled by default.
ALLOW_DOWNLOADS = True
RUN_FULL_DISTANCE_MATRIX = False
RUN_REAL_NETWORK_SMOKE = False
SMOKE_DESTINATION_LIMIT = 80
SMOKE_CANDIDATE_LIMIT = 8
SMOKE_MAX_TOTAL_DIST_M = 50_000

pd.DataFrame([asdict(CFG)])

## Shared Cache And Source Resolution

The README states that this clone uses:

`C:\local\Download_Depot\indonesia_nusa_tenggara_data`

The PBF is the shared source artefact for the new pipeline and the original implementation. The cell below resolves that artefact with `CountryDataSources` and only downloads it if it is missing and `ALLOW_DOWNLOADS` is enabled.

In [ ]:
sources = CountryDataSources(
    country_slug=CFG.country_slug,
    iso3=CFG.iso3,
    base_dir=CFG.base_dir,
    geofabrik_region=CFG.geofabrik_region,
    pbf_filename=CFG.pbf_filename,
    worldpop_filename="idn_ppp_2020.tif",  # Not used in this table-destination replication.
)

pbf_path = sources.pbf_path
if not pbf_path.exists() and ALLOW_DOWNLOADS:
    download_if_missing(sources.pbf_url, pbf_path)

cache_status = pd.DataFrame([
    {"artefact": "destination_table", "path": str(CFG.destination_table), "exists": CFG.destination_table.exists(), "bytes": CFG.destination_table.stat().st_size if CFG.destination_table.exists() else None},
    {"artefact": "osm_pbf", "path": str(pbf_path), "exists": pbf_path.exists(), "bytes": pbf_path.stat().st_size if pbf_path.exists() else None},
    {"artefact": "base_dir", "path": str(CFG.base_dir), "exists": CFG.base_dir.exists(), "bytes": None},
])
cache_status

## Load And Validate The Destination School Table

The README says the destination table is `17_routing_targets_enhanced.csv` and expects these columns:

- `source_id`
- `routing_lon`
- `routing_lat`

The precheck below reproduces the PowerShell count from the README and then filters to the configured bbox.

In [ ]:
def load_destination_table(config: DistanceReplicationConfig) -> pd.DataFrame:
    required = {config.destination_id_column, config.destination_lon_column, config.destination_lat_column}
    df = pd.read_csv(config.destination_table)
    missing = required.difference(df.columns)
    if missing:
        raise ValueError(f"Destination table is missing required columns: {sorted(missing)}")
    out = df.copy()
    out[config.destination_lon_column] = pd.to_numeric(out[config.destination_lon_column], errors="coerce")
    out[config.destination_lat_column] = pd.to_numeric(out[config.destination_lat_column], errors="coerce")
    return out

def target_contract(destinations: pd.DataFrame, config: DistanceReplicationConfig) -> pd.DataFrame:
    min_lon, min_lat, max_lon, max_lat = config.bbox
    valid = destinations.dropna(subset=[config.destination_lon_column, config.destination_lat_column]).copy()
    valid = valid[
        valid[config.destination_lon_column].between(min_lon, max_lon)
        & valid[config.destination_lat_column].between(min_lat, max_lat)
    ].copy()
    valid["target_id"] = valid[config.destination_id_column].astype(str)
    valid["target_type"] = "table"
    valid["lon"] = valid[config.destination_lon_column].astype(float)
    valid["lat"] = valid[config.destination_lat_column].astype(float)
    return valid[["target_id", "target_type", "lon", "lat"] + [c for c in ["source_name", "quality_class", "routing_target_source"] if c in valid.columns]]

destination_raw = load_destination_table(CFG)
targets = target_contract(destination_raw, CFG)

destination_checks = pd.DataFrame([
    {"check": "raw rows", "value": len(destination_raw)},
    {"check": "non-null routing lon/lat", "value": int(destination_raw[[CFG.destination_lon_column, CFG.destination_lat_column]].notna().all(axis=1).sum())},
    {"check": "targets inside bbox", "value": len(targets)},
    {"check": "unique target IDs inside bbox", "value": targets["target_id"].nunique()},
])
destination_checks

## Generate Candidate Facility Points On The 2 km Grid

The README uses `--sources candidates` and `--candidate-grid-spacing-m 2000`. This implementation creates a reproducible metric grid over the bbox using the configured metric CRS, then transforms the points back to WGS84 longitude/latitude and tags them as `source_type='candidates'`.

The original production pipeline may apply additional land, island, or road-snap filtering. This notebook keeps those filters explicit: the next step snaps candidates to the road graph and removes candidates more than 1 km from the nearest road node, matching `--candidate-max-snap-dist-m 1000`.

In [ ]:
def generate_candidate_grid(config: DistanceReplicationConfig) -> pd.DataFrame:
    min_lon, min_lat, max_lon, max_lat = config.bbox
    to_metric = Transformer.from_crs("EPSG:4326", config.metric_crs, always_xy=True)
    to_wgs84 = Transformer.from_crs(config.metric_crs, "EPSG:4326", always_xy=True)
    xs, ys = [], []
    for lon, lat in [(min_lon, min_lat), (min_lon, max_lat), (max_lon, min_lat), (max_lon, max_lat)]:
        x, y = to_metric.transform(lon, lat)
        xs.append(x)
        ys.append(y)
    x0, x1 = min(xs), max(xs)
    y0, y1 = min(ys), max(ys)
    rows = []
    candidate_index = 0
    x = x0
    while x <= x1:
        y = y0
        while y <= y1:
            lon, lat = to_wgs84.transform(x, y)
            if min_lon <= lon <= max_lon and min_lat <= lat <= max_lat:
                rows.append({
                    "source_id": f"candidate_{candidate_index:06d}",
                    "source_type": "candidates",
                    "lon": float(lon),
                    "lat": float(lat),
                })
                candidate_index += 1
            y += config.candidate_grid_spacing_m
        x += config.candidate_grid_spacing_m
    return pd.DataFrame(rows)

candidate_grid = generate_candidate_grid(CFG)
pd.DataFrame([{
    "candidate_grid_spacing_m": CFG.candidate_grid_spacing_m,
    "candidate_count_before_snap_filter": len(candidate_grid),
    "bbox": CFG.bbox,
}])

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.8))
sample_targets = targets.sample(min(len(targets), 1200), random_state=7)
sample_candidates = candidate_grid.sample(min(len(candidate_grid), 1200), random_state=7)
ax.scatter(sample_candidates["lon"], sample_candidates["lat"], s=4, c="#1b6ca8", alpha=0.35, label="2 km candidate grid sample")
ax.scatter(sample_targets["lon"], sample_targets["lat"], s=5, c="#f0b429", alpha=0.5, label="school routing targets sample")
min_lon, min_lat, max_lon, max_lat = CFG.bbox
ax.plot([min_lon, max_lon, max_lon, min_lon, min_lon], [min_lat, min_lat, max_lat, max_lat, min_lat], color="#d64545", linewidth=1.3, label="README bbox")
ax.set_title("Nusa Tenggara README geometry precheck")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.legend(frameon=False, loc="upper left")
ax.grid(True, color="#e6e8e6")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()

## Road Network Loading And Snapping Helpers

The full run loads `nusa-tenggara-latest.osm.pbf` from the shared cache with the new pipeline's `load_driving_network()`. Parsing the PBF can take a few minutes, so the notebook does not do it unless `RUN_FULL_DISTANCE_MATRIX` or `RUN_REAL_NETWORK_SMOKE` is enabled.

Snapping is done against the road-network node table. Candidate sources with snap distance greater than 1 km are removed, matching the README option `--candidate-max-snap-dist-m 1000`.

In [ ]:
def add_metric_xy(df: pd.DataFrame, lon_col: str = "lon", lat_col: str = "lat", crs: str = CFG.metric_crs) -> pd.DataFrame:
    transformer = Transformer.from_crs("EPSG:4326", crs, always_xy=True)
    x, y = transformer.transform(df[lon_col].to_numpy(), df[lat_col].to_numpy())
    out = df.copy()
    out["x_m"] = x
    out["y_m"] = y
    return out

def snap_to_network_nodes(points: pd.DataFrame, network: NetworkData, *, max_snap_dist_m: float | None = None) -> pd.DataFrame:
    nodes_xy = add_metric_xy(network.nodes[["node_id", "lon", "lat"]])
    points_xy = add_metric_xy(points)
    tree = cKDTree(nodes_xy[["x_m", "y_m"]].to_numpy())
    distances, indices = tree.query(points_xy[["x_m", "y_m"]].to_numpy(), k=1)
    snapped = points.copy()
    snapped["nearest_node"] = nodes_xy.iloc[indices]["node_id"].to_numpy()
    snapped["snap_dist_m"] = distances.astype(float)
    if max_snap_dist_m is not None:
        snapped = snapped[snapped["snap_dist_m"] <= max_snap_dist_m].copy()
    return snapped

def load_readme_network(config: DistanceReplicationConfig = CFG) -> NetworkData:
    if not pbf_path.exists():
        raise FileNotFoundError(f"Missing OSM PBF at {pbf_path}; enable downloads or rebuild the shared cache first.")
    return load_driving_network(pbf_path, bbox=config.bbox)

print("Network load helpers ready. Full network loading is guarded by RUN_FULL_DISTANCE_MATRIX / RUN_REAL_NETWORK_SMOKE.")

## Sparse Cutoff Matrix Writer

The original README writes split matrix outputs such as:

`distance_matrix_src_candidates_dst_table_<run_tag>.parquet`

The helper below writes exactly that layer family for candidate sources to table destinations. It computes shortest paths with a cutoff and streams chunks to Parquet instead of building a large all-pairs matrix in memory.

In [ ]:
def build_networkx_graph(network: NetworkData) -> nx.DiGraph:
    graph = nx.DiGraph()
    for row in network.nodes.itertuples(index=False):
        graph.add_node(row.node_id, lon=float(row.lon), lat=float(row.lat))
    for row in network.edges.itertuples(index=False):
        graph.add_edge(row.u, row.v, length_m=float(row.length_m))
    return graph

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

def write_parquet_stream(chunks: Iterable[pd.DataFrame], path: Path) -> dict[str, int | str]:
    path.parent.mkdir(parents=True, exist_ok=True)
    writer = None
    row_count = 0
    try:
        for chunk in chunks:
            if chunk.empty:
                continue
            table = pa.Table.from_pandas(chunk, preserve_index=False)
            if writer is None:
                writer = pq.ParquetWriter(path, table.schema)
            writer.write_table(table)
            row_count += len(chunk)
    finally:
        if writer is not None:
            writer.close()
    if writer is None:
        empty_schema = pa.schema([
            ("source_id", pa.string()), ("target_id", pa.string()),
            ("source_type", pa.string()), ("target_type", pa.string()),
            ("source_node", pa.int64()), ("target_node", pa.int64()),
            ("network_dist", pa.float64()), ("source_snap_dist_m", pa.float64()),
            ("target_snap_dist_m", pa.float64()), ("total_dist", pa.float64()),
        ])
        pq.write_table(pa.Table.from_pylist([], schema=empty_schema), path)
    return {"path": str(path), "rows": row_count, "bytes": path.stat().st_size, "sha256": sha256_file(path)}

def candidate_table_matrix_chunks(
    graph: nx.DiGraph,
    snapped_sources: pd.DataFrame,
    snapped_targets: pd.DataFrame,
    *,
    max_total_dist_m: float,
) -> Iterable[pd.DataFrame]:
    target_groups = {
        node: group.copy()
        for node, group in snapped_targets.groupby("nearest_node", sort=False)
    }
    target_nodes = set(target_groups)
    for source in snapped_sources.itertuples(index=False):
        cutoff = max_total_dist_m + float(getattr(source, "source_snap_dist_m", getattr(source, "snap_dist_m", 0.0)))
        lengths = nx.single_source_dijkstra_path_length(graph, source.nearest_node, cutoff=cutoff, weight="length_m")
        rows = []
        for target_node in target_nodes.intersection(lengths.keys()):
            network_dist = float(lengths[target_node])
            for target in target_groups[target_node].itertuples(index=False):
                source_snap = float(getattr(source, "snap_dist_m", 0.0))
                target_snap = float(getattr(target, "snap_dist_m", 0.0))
                total_dist = source_snap + network_dist + target_snap
                if total_dist <= max_total_dist_m:
                    rows.append({
                        "source_id": str(source.source_id),
                        "target_id": str(target.target_id),
                        "source_type": str(getattr(source, "source_type", "candidates")),
                        "target_type": str(getattr(target, "target_type", "table")),
                        "source_node": int(source.nearest_node),
                        "target_node": int(target.nearest_node),
                        "network_dist": network_dist,
                        "source_snap_dist_m": source_snap,
                        "target_snap_dist_m": target_snap,
                        "total_dist": total_dist,
                    })
        yield pd.DataFrame(rows)

def write_run_manifest(path: Path, payload: dict) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        yaml.safe_dump(payload, f, sort_keys=False)
    return path

print("Streaming split-matrix writer ready.")

## Optional Real-Network Smoke Run

Set `RUN_REAL_NETWORK_SMOKE = True` to parse the real PBF, snap a small sample of candidates and destinations, compute a small cutoff distance matrix, and write a smoke Parquet + manifest under:

`C:\local\Download_Depot\indonesia_nusa_tenggara_data\outputs\notebook_smoke`

This uses the same functions as the full run, only with small row limits and a 50 km cutoff.

In [ ]:
if RUN_REAL_NETWORK_SMOKE:
    smoke_network = load_readme_network(CFG)
    smoke_sources_raw = candidate_grid.head(SMOKE_CANDIDATE_LIMIT).copy()
    smoke_targets_raw = targets.head(SMOKE_DESTINATION_LIMIT).copy()
    smoke_sources = snap_to_network_nodes(smoke_sources_raw, smoke_network, max_snap_dist_m=CFG.candidate_max_snap_dist_m).rename(columns={"snap_dist_m": "source_snap_dist_m"})
    smoke_targets = snap_to_network_nodes(smoke_targets_raw, smoke_network).rename(columns={"snap_dist_m": "target_snap_dist_m"})
    smoke_graph = build_networkx_graph(smoke_network)
    smoke_path = SMOKE_OUTPUT_DIR / f"distance_matrix_src_candidates_dst_table_smoke_{CFG.run_tag}.parquet"
    smoke_stats = write_parquet_stream(
        candidate_table_matrix_chunks(smoke_graph, smoke_sources, smoke_targets, max_total_dist_m=SMOKE_MAX_TOTAL_DIST_M),
        smoke_path,
    )
    smoke_manifest = {
        "mode": "smoke",
        "readme": r"C:\local\GIT\Public-Infrastructure-Service-Access-remove-wfpexplaining\README_distance_replication.txt",
        "config": {k: str(v) if isinstance(v, Path) else v for k, v in asdict(CFG).items()},
        "limits": {"sources": SMOKE_CANDIDATE_LIMIT, "destinations": SMOKE_DESTINATION_LIMIT, "max_total_dist_m": SMOKE_MAX_TOTAL_DIST_M},
        "counts": {"snapped_sources": len(smoke_sources), "snapped_targets": len(smoke_targets)},
        "matrix": smoke_stats,
    }
    smoke_manifest_path = write_run_manifest(SMOKE_OUTPUT_DIR / f"run_manifest_smoke_{CFG.run_tag}.yaml", smoke_manifest)
    display(pd.DataFrame([smoke_stats]))
    print(smoke_manifest_path)
else:
    print("RUN_REAL_NETWORK_SMOKE is False; skipped real PBF parsing and smoke matrix computation.")

## Full README Distance Run

Set `RUN_FULL_DISTANCE_MATRIX = True` to execute the full distance replication.

Expected output family:

- `sources_<run_tag>.parquet`
- `targets_<run_tag>.parquet`
- `distance_matrix_src_candidates_dst_table_<run_tag>.parquet`
- `run_manifest_<run_tag>.yaml`

The full run may be expensive. It parses the PBF, snaps the full candidate grid and all valid school destinations, and computes candidate-to-school shortest paths with a 150 km cutoff.

In [ ]:
def run_full_readme_distance_replication(config: DistanceReplicationConfig = CFG) -> dict:
    output_dir = config.base_dir / "outputs"
    output_dir.mkdir(parents=True, exist_ok=True)
    network = load_readme_network(config)
    snapped_sources = snap_to_network_nodes(candidate_grid, network, max_snap_dist_m=config.candidate_max_snap_dist_m).rename(columns={"snap_dist_m": "source_snap_dist_m"})
    snapped_targets = snap_to_network_nodes(targets, network).rename(columns={"snap_dist_m": "target_snap_dist_m"})

    sources_path = output_dir / f"sources_{config.run_tag}.parquet"
    targets_path = output_dir / f"targets_{config.run_tag}.parquet"
    snapped_sources.to_parquet(sources_path, index=False)
    snapped_targets.to_parquet(targets_path, index=False)

    graph = build_networkx_graph(network)
    matrix_path = output_dir / f"distance_matrix_src_candidates_dst_table_{config.run_tag}.parquet"
    matrix_stats = write_parquet_stream(
        candidate_table_matrix_chunks(graph, snapped_sources, snapped_targets, max_total_dist_m=config.max_total_dist_m),
        matrix_path,
    )

    manifest = {
        "mode": "full_readme_replication",
        "readme": r"C:\local\GIT\Public-Infrastructure-Service-Access-remove-wfpexplaining\README_distance_replication.txt",
        "config": {k: str(v) if isinstance(v, Path) else v for k, v in asdict(config).items()},
        "sources": {"path": str(sources_path), "rows": len(snapped_sources), "bytes": sources_path.stat().st_size, "sha256": sha256_file(sources_path)},
        "targets": {"path": str(targets_path), "rows": len(snapped_targets), "bytes": targets_path.stat().st_size, "sha256": sha256_file(targets_path)},
        "matrix": matrix_stats,
        "network": {"pbf_path": str(pbf_path), "nodes": len(network.nodes), "edges": len(network.edges)},
    }
    manifest_path = write_run_manifest(output_dir / f"run_manifest_{config.run_tag}.yaml", manifest)
    return {"manifest_path": manifest_path, "manifest": manifest}

if RUN_FULL_DISTANCE_MATRIX:
    full_result = run_full_readme_distance_replication(CFG)
    print(full_result["manifest_path"])
    display(pd.DataFrame([full_result["manifest"]["matrix"]]))
else:
    print("RUN_FULL_DISTANCE_MATRIX is False; full README distance run was not launched.")

## Facility-Location And TSP Continuation

The updated README adds the missing next code layer. After the distance matrix exists, the workflow continues as:

1. Load the candidate-to-school matrix and demand table.
2. Solve the uncapacitated facility-location model.
3. Build one assigned-school TSP per selected facility.
4. Plot road-network legs and mark disconnected fallback legs.

For a complete miniature formulation, see:

`wfp_location_routing_experiment.ipynb`

For production route plotting from GeoJSON/parquet outputs, the README points to:

`general_distances_per_country\distance_pipeline\viz.py`

In [ ]:
def summarize_existing_outputs(output_dir: Path = OUTPUT_DIR) -> pd.DataFrame:
    if not output_dir.exists():
        return pd.DataFrame(columns=["name", "bytes", "last_modified"])
    rows = []
    for path in sorted(output_dir.glob("*")):
        if path.is_file():
            rows.append({"name": path.name, "bytes": path.stat().st_size, "last_modified": pd.Timestamp(path.stat().st_mtime, unit="s")})
    return pd.DataFrame(rows).sort_values("last_modified", ascending=False) if rows else pd.DataFrame(columns=["name", "bytes", "last_modified"])

summarize_existing_outputs().head(20)

## Quality Checklist From The README

Use this checklist after running either the smoke or full distance job:

- Destination row count matches the intended school subset; full run target is 14,101 geocoded rows in the latest notes.
- `quality_class`, `needs_review`, and routing-target fields are inspected in the chosen geocoded table.
- Matrix row count is non-zero and plausible for the cutoff.
- A manifest exists and records row counts, file sizes, hashes, paths, bbox, candidate spacing, snap threshold, and max distance.
- If map generation is added, the map file is stored under the same output family.
- Facility-location and TSP notebooks consume the manifest/run tag rather than guessing filenames.

In [ ]:
quality_check_template = pd.DataFrame([
    {"check": "destination table exists", "status": CFG.destination_table.exists(), "detail": str(CFG.destination_table)},
    {"check": "OSM PBF exists in shared cache", "status": pbf_path.exists(), "detail": str(pbf_path)},
    {"check": "non-null routing targets", "status": int(destination_checks.loc[destination_checks['check'] == 'non-null routing lon/lat', 'value'].iloc[0]) > 0, "detail": int(destination_checks.loc[destination_checks['check'] == 'non-null routing lon/lat', 'value'].iloc[0])},
    {"check": "candidate grid generated", "status": len(candidate_grid) > 0, "detail": len(candidate_grid)},
    {"check": "full run switch", "status": RUN_FULL_DISTANCE_MATRIX, "detail": "False means configuration/precheck notebook execution only"},
])
quality_check_template